# flux-modular · explore — run any recipe, sweep any configuration, compare
Loads the `flux_modular` primitive + `RecipeRunner`, runs the shipped **recipes** (methods-as-configs), and renders
side-by-side comparison grids for NEW configurations — including compositions and hyper-parameter sweeps the
source papers never tried. Metrics are a PANEL (depth + appearance + prompt), shown to RANK the grid; the GO/NO-GO
call is by eye (single scalars have been gamed twice — depth-corr and CLIP). Runtime: 80GB A100, FLUX.1-dev license.

In [ ]:
import subprocess, os
for _ in range(3):
    if subprocess.call(["pip","install","-q","git+https://github.com/huggingface/diffusers.git"])==0: break
!pip install -q transformers accelerate sentencepiece protobuf hf_transfer scikit-image pyyaml
# get the repo (flux_modular package + recipes/) — swap for your fork/branch as needed
if not os.path.isdir("flux-recipes"):
    subprocess.call(["git","clone","-q","https://github.com/remyxai/flux-recipes.git"])

In [ ]:
import sys, os, torch, numpy as np
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF","expandable_segments:True")
sys.path.insert(0, "flux-recipes")
from huggingface_hub import login
try:
    from google.colab import userdata; login(userdata.get("HUGGINGFACE_TOKEN"))
except Exception:
    login()
assert torch.cuda.is_available(); print("GPU:", torch.cuda.get_device_name(0))
from flux_modular import RecipeRunner, load_recipes
runner = RecipeRunner()                       # loads FLUX.1-dev (+ Redux lazily)
RECIPES = load_recipes("flux-recipes/recipes")
import pandas as pd
print(pd.DataFrame([{"recipe":k,"validated":v.get("validated"),"requires":",".join(v.get("requires",[])),
                     "inputs":",".join(v.get("inputs",[]))} for k,v in RECIPES.items()]).to_string(index=False))

In [ ]:
# --- metric PANEL (rank only) + comparison-grid helper ---
from transformers import pipeline as hf_pipeline, CLIPModel, CLIPProcessor
from PIL import Image, ImageDraw
from skimage import data
from IPython.display import display
_dep=hf_pipeline("depth-estimation", model="depth-anything/Depth-Anything-V2-Small-hf", device=0)
def depth_corr(a,b):
    da=np.asarray(_dep(a)["depth"].convert("L").resize((256,256)),np.float32).ravel()
    db=np.asarray(_dep(b)["depth"].convert("L").resize((256,256)),np.float32).ravel()
    return float(np.corrcoef(da,db)[0,1])
_clip=CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to("cuda").eval(); _cp=CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
@torch.no_grad()
def clip_img(a,b):
    px=_cp(images=[a,b],return_tensors="pt").to("cuda")
    v=_clip.vision_model(pixel_values=px["pixel_values"]).pooler_output; e=_clip.visual_projection(v); e=e/e.norm(dim=-1,keepdim=True)
    return float((e[0]@e[1]).cpu())
@torch.no_grad()
def clip_txt(im,txt):
    px=_cp(images=[im],return_tensors="pt").to("cuda"); ti=_cp(text=[txt],return_tensors="pt",padding=True).to("cuda")
    o=_clip(pixel_values=px["pixel_values"],input_ids=ti["input_ids"],attention_mask=ti["attention_mask"])
    e=o.image_embeds/o.image_embeds.norm(dim=-1,keepdim=True); t=o.text_embeds/o.text_embeds.norm(dim=-1,keepdim=True)
    return float((e@t.T).squeeze().cpu())
def compare(panels, cellpx=300):
    # panels: list of (label, image, sublabel-or-None)
    g=Image.new("RGB",(len(panels)*cellpx+(len(panels)+1)*6, cellpx+34),"white"); d=ImageDraw.Draw(g)
    for j,(name,im,sub) in enumerate(panels):
        x=6+j*(cellpx+6); g.paste(im.resize((cellpx,cellpx)),(x,4)); d.text((x+4,cellpx+8),str(name)[:34],fill="black")
        if sub: d.text((x+4,cellpx+20),str(sub)[:34],fill="black")
    display(g)
# demo inputs
REF_S=Image.fromarray(data.astronaut()).convert("RGB").resize((1024,1024))
REF_A=Image.fromarray(data.coffee()).convert("RGB").resize((1024,1024))
INP={"prompt":"a portrait of a person","ref_structure":REF_S,"ref_appearance":REF_A}
print("metrics + refs ready")

## 1. Run a recipe (each shipped method = one config)

In [ ]:
fc = runner.run(RECIPES["freecontrol"], INP, S=0.3)
compare([("ref_structure",REF_S,None), ("freecontrol S=0.3", fc, f"depth-corr={depth_corr(fc,REF_S):.2f}")])

### ...and a different op class from a config — `regional` (bias mask, no ref image)

In [ ]:
REG_INP = {"base_prompt": "a cozy living room, soft light",
           "regions": [{"prompt": "a red velvet sofa", "bbox": [0.0, 0.45, 0.55, 1.0]},
                       {"prompt": "a tall green potted plant", "bbox": [0.6, 0.2, 1.0, 1.0]}]}
reg = runner.run(RECIPES["regional"], REG_INP, seed=0)
# CLIP-to-region-prompt as a rough presence check (rank only)
compare([("regional (2 regions)", reg,
          f"sofa={clip_txt(reg,'a red sofa'):.2f} plant={clip_txt(reg,'a green plant'):.2f}")])
print("regional = the `bias` op (position-agnostic) driven from a config; no capture, no ref image.")

## 2. Sweep a NEW configuration — 2-D probe over a composition (structure_appearance)

In [ ]:
cells_, recs = runner.sweep(RECIPES["structure_appearance"], {"S":[0.3,0.5], "redux_scale":[0.5,1.0,1.5]}, INP)
panels=[("ref_structure",REF_S,None),("ref_appearance",REF_A,None)]
rows=[]
for (label,im,ov) in cells_:
    d,a = depth_corr(im,REF_S), clip_img(im,REF_A)
    panels.append((label, im, f"d={d:.2f} a={a:.2f}")); rows.append({**ov,"depth_corr":round(d,3),"clip_appear":round(a,3)})
# render as two rows (by S) for readability
byS={}
for p,r in zip(panels[2:],rows): byS.setdefault(r["S"],[]).append(p)
compare(panels[:2])
for S,ps in byS.items(): print(f"S={S}"); compare(ps)
import pandas as pd; print(pd.DataFrame(rows).to_string(index=False))
print("\nRANK by the panel; pick the GO cell by EYE — high depth-corr with a washed image is a metric artifact,")
print("not structure (seen twice). This 2-D (cutoff x appearance-strength) neighbourhood is one no paper tried.")

## 3. Editing + consistency families — different run-loops, same interface
`run: edit` (RF-inversion → substitute) and `run: batch` (frame-shared K/V) go through the same `runner.run`.
These recipes are `expressible` (faithful ports, not yet GPU-cross-checked) — this cell IS the validation.

In [ ]:
# --- kv-edit (editing): keep background, change the subject inside a mask ---
from PIL import Image as _I, ImageDraw as _D
src = REF_S   # astronaut as the source image
mask = _I.new("L",(1024,1024),0); _D.Draw(mask).ellipse([300,120,760,760], fill=255)  # white=edit (the figure), black=keep
kve = runner.run(RECIPES["kv-edit"], {"image": src, "source_prompt": "a portrait of an astronaut",
                                    "prompt": "a portrait of a bronze knight in armor", "mask": mask}, seed=0)
cse = runner.run(RECIPES["consistedit"], {"image": src, "source_prompt": "a portrait of an astronaut",
                                        "prompt": "a portrait of a bronze knight in armor", "mask": mask},
               seed=0, consistency_strength=0.3)
compare([("source",src,None), ("edit mask",mask.convert("RGB"),None),
         ("kv-edit (substitute)",kve,"bg kept?"), ("consistedit (blend)",cse,"structure kept?")])

# --- story (consistency): one character across scenes (returns a list of frames) ---
frames = runner.run(RECIPES["story"], {"character_prompt":"a young woman with red hair and freckles",
                                     "scene_prompts":["reading in a cafe #cafe","walking in a forest #forest",
                                                      "[NC] an empty street at night #street"]}, seed=0)
compare([(f"frame {i}", im, None) for i,im in enumerate(frames)])
print("kv-edit: op=substitute on the edit run-loop; story: op=share on the batch run-loop — same runner.run().")

## 4. New method variants — authored as configs, no new code
A method is a dict, so a *variant* is a tweaked dict. Two examples the shipped recipes don't cover.

In [ ]:
import copy
PROMPT = "a bronze knight in armor"
# variant A — 'freecontrol-loose': fewer late blocks + shorter injection = looser structure, more prompt freedom
loose = copy.deepcopy(RECIPES["freecontrol"]); loose["params"] = {**loose["params"], "S": 0.15, "last_n": 12}
tight_im = runner.run("freecontrol", {"prompt": PROMPT, "ref_structure": REF_S}, S=0.3, seed=0)
loose_im = runner.run(loose,          {"prompt": PROMPT, "ref_structure": REF_S},        seed=0)
compare([("ref_structure", REF_S, None),
         ("freecontrol (tight S=0.3)", tight_im, f"d={depth_corr(tight_im,REF_S):.2f}"),
         ("freecontrol-loose (new)",  loose_im,  f"d={depth_corr(loose_im,REF_S):.2f}")])
print("A new method point is a new dict — lower S + fewer blocks = looser structure / more prompt freedom.")

### Variant B — appearance strength as a knob (`redux_scale`)
Same `appearance` recipe, one knob: `redux_scale` dials how strongly the reference material is applied
(confirmed monotonic in the §2 sweep). Subtle vs strong material transfer, no new code.

In [ ]:
subtle = runner.run("appearance", {"prompt": "a portrait of a person", "ref_appearance": REF_A}, redux_scale=0.5, seed=0)
strong = runner.run("appearance", {"prompt": "a portrait of a person", "ref_appearance": REF_A}, redux_scale=2.0, seed=0)
compare([("ref_appearance", REF_A, None),
         ("appearance redux=0.5 (subtle)", subtle, f"a={clip_img(subtle,REF_A):.2f}"),
         ("appearance redux=2.0 (strong)", strong, f"a={clip_img(strong,REF_A):.2f}")])
print("Same recipe, redux_scale dials material strength — a config knob, subtle vs strong appearance.")

## 5. New compositions — capabilities no single recipe has
Two ways to compose: **one pass** (multiple ops in one denoise) and **chaining** (one recipe's output feeds the next).

### Single-pass — `structure_regional` = freecontrol ⊕ regional (both hooks in one payload)
Keep the reference's layout (Q-replace) *and* route a different prompt per region (bias) — in one denoise.

In [ ]:
SR_IN = {"ref_structure": REF_S, "base_prompt": "a museum bust",
         "regions": [{"prompt": "a bronze robot bust",       "bbox": [0.0, 0.0, 0.5, 1.0]},
                     {"prompt": "a white marble statue bust", "bbox": [0.5, 0.0, 1.0, 1.0]}]}
sr = runner.run("structure_regional", SR_IN, S=0.3, seed=0)
compare([("ref_structure", REF_S, None),
         ("structure_regional", sr, f"d={depth_corr(sr,REF_S):.2f}  bronze L / marble R")])
print("One denoise: reference layout (pre_rope Q-replace) + per-region prompts (bias) — a composition no single")
print("recipe or paper has. Left half bronze robot, right half marble statue, in the reference's composition.")

### Chaining — one recipe's output is the next one's input

In [ ]:
sa = runner.run("structure_appearance",
                {"prompt": "a portrait of a person", "ref_structure": REF_S, "ref_appearance": REF_A}, S=0.5, seed=0)
crown_mask = Image.new("L", (1024, 1024), 0); ImageDraw.Draw(crown_mask).ellipse([360, 90, 690, 470], fill=255)  # head
chained = runner.run("kv-edit",
                     {"image": sa, "source_prompt": "a portrait", "prompt": "a portrait wearing a golden crown",
                      "mask": crown_mask}, seed=0)
compare([("structure_appearance", sa, None), ("+ kv-edit: add a crown", chained, "chained recipes")])
print("Recipes chain: structure_appearance's output becomes kv-edit's input image — composition, no new code.")

### Reference-conditioned story — a character from a photo across scenes (story ⊕ appearance)
`reference_image` Redux-conditions every frame; the K/V share keeps it consistent. Appearance-level (the look),
**not** tight face identity — that's the identity/residual path, still to come.

In [ ]:
frames = runner.run("story_reference",
                    {"reference_image": REF_S,   # the character reference (a photo)
                     "scene_prompts": ["reading a book in a cafe #cafe", "walking through a forest #forest"]},
                    seed=0)
compare([("reference", REF_S, None)] + [(f"scene {i}", im, None) for i, im in enumerate(frames)])
print("story_reference = Redux(reference) on each frame + cross-frame K/V share -> the reference's look carries")
print("across scenes. For a face-LOCK (not just the look), that's the identity/residual path, next on the list.")

### Multi-reference appearance — mix two materials (declarative, sweepable scales)
`appearance_mix` lists **two Redux donors**; each scale is overridable per-run (`redux_scale_ref_a` / `_ref_b`),
so a donor drops out at 0. Material mixing as data, not a hand-merge.

In [ ]:
from skimage import data as _d
REF_B2 = Image.fromarray(_d.chelsea()).convert("RGB").resize((1024, 1024))   # a 2nd material (tabby fur)
mix_in = {"prompt": "a ceramic vase", "ref_a": REF_A, "ref_b": REF_B2}
aonly = runner.run("appearance_mix", mix_in, redux_scale_ref_b=0.0, seed=0)   # donor B off -> A only
bonly = runner.run("appearance_mix", mix_in, redux_scale_ref_a=0.0, seed=0)   # donor A off -> B only
mix   = runner.run("appearance_mix", mix_in, seed=0)                          # both
compare([("ref_a (coffee)", REF_A, None), ("ref_b (fur)", REF_B2, None),
         ("A only", aonly, None), ("B only", bonly, None), ("mix A+B", mix, None)])
print("Two Redux donors with independent, sweepable scales — declarative `sources` list, not a hand-merge.")

### Residual / feature hook — the identity seam (`reference_echo`)
A different primitive: forward-hooks that modulate a block's OUTPUT (not attention q/k/v). Here a training-free
**feature echo** reconstructs the reference — a strong pull that overrides the prompt at higher strength. This is
the seam a trained **identity** adapter (PuLID) plugs into for face-lock; the echo is the mechanism proof.

In [ ]:
base_re = runner.run("freecontrol", {"prompt": "a bronze statue bust", "ref_structure": REF_S}, S=0.0, seed=0)  # ~plain gen
echo    = runner.run("reference_echo", {"prompt": "a bronze statue bust", "ref_structure": REF_S}, S=0.3, seed=0)
compare([("ref_structure", REF_S, None),
         ("plain gen (no hook)", base_re, f"d={depth_corr(base_re,REF_S):.2f}"),
         ("reference_echo (residual)", echo, f"d={depth_corr(echo,REF_S):.2f}")])
print("Residual feature-echo pulls the output toward the reference (strong). The op menu now spans attention AND")
print("block-output residuals — the seam identity/PuLID injection uses. Face-lock = a trained ID adapter here, next.")

## 6. Your turn — edit a recipe inline and compare
`load_recipes` returns plain dicts; override any field, or pass `**overrides` to `run`/`sweep`. Add new YAMLs to
`recipes/` (mark `validated: expressible`) and re-run to bring more methods into the comparison.

In [ ]:
# example: same composition, weaker structure lock + stronger appearance, one-off
img = runner.run(RECIPES["structure_appearance"], INP, S=0.4, redux_scale=2.0, seed=1)
compare([("ref_structure",REF_S,None),("ref_appearance",REF_A,None),
         ("S=0.4 redux=2.0", img, f"d={depth_corr(img,REF_S):.2f} a={clip_img(img,REF_A):.2f}")])